# Limenex + LangGraph: OpenAI Credit Top-Ups

This notebook shows a minimal **before/after** example:

- **Before**: a naive tool that lets an agent top up its own OpenAI API credits with no limits.
- **After**: the same action wired through Limenex, with hard **BLOCK** and soft **ESCALATE** limits.

We mock the "agent" by calling functions directly — no real LLM or OpenAI API key is required.

In [ ]:
# If you're running this outside the repo environment, you may need:
# !pip install limenex

In [ ]:
from pathlib import Path

from limenex.core.engine import BlockedError, EscalationRequired, PolicyEngine
from limenex.core.policy_store import LocalFilePolicyStore
from limenex.core.stores import LocalFileStateStore
from limenex.skills.finance import make_spend

## 1. Before: naive top-ups (no governance)

This is essentially what a naive tool would look like: the agent can call it whenever it wants,
with whatever amount it wants. There are no limits, no log, and no approval path.

In [ ]:
# BEFORE: unguarded "top up OpenAI credits" helper

async def naive_top_up_openai(amount_usd: float) -> None:
    print(f"[naive] Topped up OpenAI credits by ${amount_usd:.2f}")

async def naive_scenario() -> None:
    print("=== BEFORE: naive top-ups ===")
    await naive_top_up_openai(40.0)   # looks fine
    await naive_top_up_openai(600.0)  # just as easy

await naive_scenario()

### “Why not just write an `if` in the function?”

You absolutely *can* hard‑code checks like:

```python
async def naive_top_up_openai(amount_usd: float) -> None:
    if amount_usd > 500:
        raise ValueError("too much")
    ...
```

That works for a single function in a single service.

Limenex is solving a different problem:

- **Policies live in data**, not code (`policies.yaml`), so ops / security can change limits without redeploying agents.
- **One engine enforces all rules** across many skills and agents, instead of copy‑pasted `if` blocks scattered through your codebase.
- **Verdicts are consistent and auditable** (ALLOW / ESCALATE / BLOCK) with state tracked per agent and dimension, not ad‑hoc exceptions.

The rest of this notebook shows how those policies are defined once and applied everywhere the skill is used.

## 2. After: Limenex-governed top-ups

We define two deterministic policies for `finance.spend`:

1. **BLOCK** any single top-up over \$500 — a **stateless** per-call check (no approval path).
2. **ESCALATE** when cumulative spend for this agent would exceed \$100 — a **stateful** cumulative check (human review).

We store policies in `.limenex/policies_langgraph.yaml` and state in `.limenex/state_langgraph.json`.

In [ ]:
limenex_dir = Path(".limenex")
limenex_dir.mkdir(exist_ok=True)

policies_path = limenex_dir / "policies_langgraph.yaml"
policies_path.write_text(
    """
finance.spend:
  policies:
    # 1) Hard-block any single top-up over $500 — no approval path
    - type: deterministic
      dimension: openai_single_topup_usd
      operator: lt
      value: 500.0
      param: amount_usd
      breach_verdict: BLOCK
      stateful: false

    # 2) Escalate if cumulative spend would exceed $100 — human review
    - type: deterministic
      dimension: openai_cumulative_spend_usd
      operator: lt
      value: 100.0
      param: amount_usd
      breach_verdict: ESCALATE
""".strip(),
    encoding="utf-8",
)

print(policies_path.read_text())

### Policy order is safety‑critical

Policies are evaluated in the order you define them, and the engine stops on the first non‑ALLOW verdict. This means earlier rules can prevent later ones from ever running.

In particular, putting an escalation policy *before* a strict block policy is dangerous: a request that should be blocked outright might instead be routed into a human‑approval flow.

As a rule of thumb, keep hard safety rules (like BLOCK limits and absolute caps) before softer behaviours (like ESCALATE or log‑only), and avoid long chains of overlapping policies so behaviour stays predictable.

In [ ]:
# Set up Limenex engine and the governed finance.spend skill

policy_store = LocalFilePolicyStore(policies_path)
state_store = LocalFileStateStore(limenex_dir / "state_langgraph.json")
engine = PolicyEngine(policy_store=policy_store, state_store=state_store)

# Payment executor — this would be your real OpenAI billing call in production
async def openai_top_up_executor(amount_usd: float) -> None:
    print(f"[governed] Approved OpenAI top-up of ${amount_usd:.2f}", flush=True)

spend = make_spend(engine, registry={"openai": openai_top_up_executor})

# This is what you'd expose to your LLM agent as a tool — plain data parameters only:
# async def top_up_openai(amount_usd: float) -> None: ...
AGENT_ID = "research-agent-1"

async def top_up_openai(amount_usd: float) -> None:
    # agent_id is application-layer context, not agent input
    await spend(agent_id=AGENT_ID, service="openai", amount_usd=amount_usd)

### Governed scenario

We simulate four calls:

1. Top up \\$40 (allowed).
2. Top up \\$30 (allowed, cumulative now \\$70).
3. Top up \\$50 (would take cumulative to \\$120) -> ESCALATE.
4. Top up \\$600 (fails the single-top-up limit) -> BLOCK.

On escalation we store the paused action, ask the human for approval in the notebook, and re-invoke the governed call if they approve.

### Escalation handling is application logic

When Limenex decides an action should be **escalated**, it raises `EscalationRequired`.
What happens next is up to your application code:

- Catch `EscalationRequired`
- Store the paused action somewhere durable (database, queue, file)
- Notify a human (Slack, email, dashboard, etc.)
- If they approve, run the action via your own code path (usually by calling the executor)
- Update the state store manually so cumulative tracking stays consistent

In this example, on approval we call `openai_top_up_executor(...)` directly
instead of going back through the governed `spend(...)` skill, so the same
request does not trigger another escalation. Because this bypasses governance,
we also call `state_store.record(...)` ourselves to keep cumulative state accurate.

In [ ]:
async def attempt_top_up(amount_usd: float) -> None:
    try:
        await top_up_openai(amount_usd)
    except EscalationRequired:
        print(f"[governed] ${amount_usd:.0f} top-up escalated — cumulative limit reached.")
        print("  → Storing paused action for human review...")

        response = input(f"Approve ${amount_usd:.0f} top-up? (y/n) ").strip().lower()
        if response == "y":
            print("[human] Approved. Re-invoking governed call...")
            await openai_top_up_executor(amount_usd=amount_usd)
            # Record state manually since we bypassed the governed path
            state_store.record(AGENT_ID, "openai_cumulative_spend_usd", amount_usd)
        else:
            print("[human] Rejected. No top-up executed.")
    except BlockedError:
        print(f"[governed] ${amount_usd:.0f} top-up blocked — exceeds single transaction limit.")


async def governed_scenario() -> None:
    print("=== AFTER: governed top-ups ===\n")

    await attempt_top_up(40.0)
    await attempt_top_up(30.0)
    await attempt_top_up(50.0)
    await attempt_top_up(600.0)

await governed_scenario()

In [ ]:
# Optional: wire the governed skill as a LangGraph tool
# Requires: pip install langchain-core langgraph

from langchain_core.tools import tool


@tool
async def top_up_openai_credits(amount_usd: float) -> str:
    """Top up OpenAI API credits. Governed by Limenex — subject to
    per-call and cumulative spend limits."""
    try:
        await spend(agent_id=AGENT_ID, service="openai", amount_usd=amount_usd)
        return f"Topped up ${amount_usd:.2f} successfully."
    except BlockedError:
        return f"Blocked — ${amount_usd:.2f} exceeds the single transaction limit."
    except EscalationRequired:
        return f"Escalated — ${amount_usd:.2f} requires human approval."

### That's the entire integration

`top_up_openai_credits` is now a standard LangGraph tool. Pass it to a `ToolNode` and bind
it to your agent's model — governance is invisible to the agent and the graph definition.

The agent sees a plain function with `amount_usd: float`. Limenex evaluates policy before
execution. No changes to your graph, your model, or your executor.

### Inspect state

Limenex stores cumulative state per `(agent_id, dimension)` pair in a JSON file. You can
inspect it at any time to understand past decisions.

In [ ]:
state_path = limenex_dir / "state_langgraph.json"
print("=== State file ===")
print(state_path.read_text())

### A note on concurrency

`LocalFileStateStore` is thread-safe within a single process via an in-process lock, and
uses atomic file writes to avoid corruption. It is **not** safe for concurrent multi-process
access (e.g. multiple worker processes sharing the same state file).

For multi-process deployments, implement the `StateStore` protocol with cross-process file
locking (e.g. `filelock`), or use a backend with atomic operations (database, Redis).

### Reset state for re-runs

Reset the state file so you can re-run the notebook from the top with a clean slate.

In [ ]:
state_path.write_text("{}", encoding="utf-8")
print("State reset.")